### Notebook to create BLOCK-T411 Exposure time test to check wavefront estimation

This test takes a set of exposures with different exposures times to study the effect on wavefront estimation.

Created on: 2025-03-27

Author: Guillem Megias

In [ ]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
import os

In [ ]:
current_path = os.getcwd()
block_number = 'T411'
program = "BLOCK-T411"
reason = "wet013_exposure_time_"
constraints = []

### Define configuration schema

In [ ]:
# Define the configurable properties that we will use in the configuration schema
properties = {
    "filter": {
        "description": "Filter to use.",
        "type": "string",
        "default": "r_57"
    },
    "maxiter": {
        "description": "Maximum number of iterations for each of the exposure times.",
        "type": "integer",
        "default": 5
    }
}

# Build the configuration schema for BLOCK-404
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

### Define scripts and block

In [ ]:
exposure_times = [5.0, 15.0, 30.0, 60.0, 120.0]
scripts = []

for exposure_time in exposure_times:
    take_image_script = ObservingScript(
        name="maintel/take_image_lsstcam.py",
        standard=True,
        parameters= dict(
            filter="$filter",
            program="$program",
            reason=f"{reason}{np.round(exposure_time, 0).astype(int)}s",
            exp_times=exposure_time,
            image_type="ACQ",
            nimages="$maxiter",
        )
    )

    scripts.append(take_image_script)

In [ ]:
block = ObservingBlock(
    name = program,
    program = program,
    configuration_schema=configuration_schema,
    scripts = scripts,
)

### Save configurable block

In [ ]:
block.model_dump_json(indent=2)

output_file_path = f'{current_path}/aos/ts_config_ocs/Scheduler/observing_blocks_maintel/AOS/LUTs/{program}.json'

with open(output_file_path, 'w') as file:
    file.write(block.model_dump_json(indent=2))